# A shared decision surface and A/B variants

Boolean, choice and score examples with local receipts. Model-judged claim consistency is **not independent verification**. Fleet selection below uses fictional profiles and performs no SSH, provisioning, account operation or remote job dispatch.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from jev_lab.safety import live_enabled
RUN_LIVE = live_enabled()
print('Live API calls:', RUN_LIVE)
# Live calls send cell inputs to TypeSafe and may incur charges.
# Explicit opt-in: export JEV_LAB_LIVE=1 and TYPESAFE_API_KEY before launching Jupyter.
from jev_lab.surface import JevDecisionSurface
SURFACE = JevDecisionSurface()
print('Decision surface ready; no API request sent.')

In [ ]:
if RUN_LIVE:
    def route_query(query):
        deep, p, rc = SURFACE.decide_bool(
            "routing", query,
            "Does answering this query well require a strong reasoning model rather than a cheap fast one?",
            threshold=0.5)
        return ("STRONG" if deep else "CHEAP"), p

    for q in ["What is 2+2?",
              "Design a fair multi-tenant rate limiter and justify the algorithm.",
              "Capital of Norway?"]:
        lane, p = route_query(q)
        print(f"  {lane:6} p={p:.2f} | {q[:55]}")
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

In [ ]:
if RUN_LIVE:
    chunks = [
        "HNSW builds a proximity graph for fast approximate nearest neighbor search.",
        "# 3.2\n[TOC] 1.Intro 2.Setup",
        "Product quantization compresses vectors to speed distance computation.",
        "um idk the vector thing is fast or whatever",
    ]
    # gate (one batched call)
    gate = SURFACE.batch_bool("retrieval", chunks,
        lambda i, c: "is this chunk substantive content worth embedding for semantic search?")
    keep = [i for i in gate if gate[i][0]]
    print(f"retrieval gate: keep {keep} drop {[i for i in gate if not gate[i][0]]}")

    # rerank the kept chunks for a query (one batched score call)
    query = "fast ANN vector search"
    for i in keep:
        sc, conf, _ = SURFACE.decide_score("retrieval",
            f"Query: {query}\nChunk: {chunks[i]}",
            "Relevance of this chunk to the query, 0=none 3=high.", [0, 1, 2, 3])
        print(f"  rerank score={sc:.2f} conf={conf:.2f} | {chunks[i][:45]}")
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

In [ ]:
if RUN_LIVE:
    # verify claimed work: is the claimed result consistent with the task?
    claims = [
        ("Task: compute 7*19. Claimed result: 133.", True),    # correct claim
        ("Task: compute 7*19. Claimed result: 130.", False),   # wrong claim
        ("Task: summarize 'cats are mammals'. Claimed: 'cats are animals'.", True),
        ("Task: summarize 'cats are mammals'. Claimed: 'cats are reptiles'.", False),
    ]
    correct = 0
    for claim_text, expected in claims:
        sound, p, rc = SURFACE.decide_bool(
            "trust", claim_text,
            "Is the claimed result a correct/consistent completion of the stated task? Answer yes only if the claim is verifiably sound.",
            threshold=0.5)
        ok = (sound == expected); correct += ok
        print(f"  {'✓' if ok else '✗'} sound={sound!s:5} p={p:.2f} | {claim_text[:52]}")
    print(f"trust verification accuracy: {correct}/{len(claims)}")
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

In [ ]:
if RUN_LIVE:
    LENSES = ["causal", "probabilistic", "premortem", "consider_opposite", "calibration", "reference_class"]
    def select_lenses(turn, threshold=0.5):
        state = "Turn: " + turn
        picked = []
        for lens in LENSES:
            fire, p, _ = SURFACE.decide_bool(
                "bias", state,
                f"Would applying the {lens} reasoning lens materially improve the answer to this turn?")
            if fire: picked.append((lens, p))
        return picked

    turn = "We've invested $2M, so we have to keep going — quitting now means it was wasted."
    sel = select_lenses(turn)
    print(f"turn: {turn[:60]}")
    print(f"lenses fired: {[(l, round(p,2)) for l,p in sel]}")
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

In [ ]:
# The variant registry: name -> config. Edit this to try new structures.
VARIANTS = [
    {"name": "routing-v1-bool", "domain": "routing", "kind": "bool",
     "instruction": "Does answering this query well require a strong reasoning model rather than a cheap fast one?",
     "threshold": 0.5},
    {"name": "routing-v2-3way", "domain": "routing", "kind": "choice",
     "options": ["cheap", "mid", "strong"],
     "instruction": "Route this query to the cheapest model tier that can answer it well: cheap (fast small model), mid (balanced), or strong (frontier reasoning)."},
    {"name": "routing-v3-score", "domain": "routing", "kind": "score",
     "options": ["0", "1", "2", "3"],
     "instruction": "Rate the reasoning complexity of this query: 0=trivial lookup, 1=simple, 2=moderate multi-step, 3=deep expert reasoning."},
    {"name": "fleet-v1-choice", "domain": "fleet", "kind": "choice",
     "options": ["node-a", "node-b", "node-c"],
     "instruction": "Which single node is the best fit to run this task right now? Consider free memory, cpu count, and current load."},
    {"name": "trust-v1-bool", "domain": "trust", "kind": "bool",
     "instruction": "Is the claimed result a correct and consistent completion of the stated task? Answer yes only if verifiably sound.",
     "threshold": 0.5},
]

# the shared test suite per domain
SUITES = {
    "routing": [
        "What is 2+2?",
        "Capital of Norway?",
        "Explain the difference between TCP and UDP.",
        "Design a fair multi-tenant rate limiter and justify the algorithm choice.",
        "Prove that the square root of 2 is irrational and discuss its historical significance.",
        "Write a one-liner to reverse a string in Python.",
    ],
    "fleet": ["Run a 6GB LLM inference batch needing low load. Nodes: node-a (4cpu 7GB free load 0.1), node-b (4cpu 5GB free load 0.2), node-c (3cpu 2.8GB free load 0.1)."],
    "trust": [
        "Task: compute 7*19. Claimed result: 133.",
        "Task: compute 7*19. Claimed result: 130.",
        "Task: name the capital of France. Claimed: Paris.",
        "Task: name the capital of France. Claimed: Lyon.",
    ],
}
print(f"{len(VARIANTS)} variants registered:")
for v in VARIANTS:
    print(f"  {v['name']:18} domain={v['domain']:9} kind={v['kind']:7}")

In [ ]:
if RUN_LIVE:
    def run_variant(variant, subjects):
        """Fire one variant config over a list of subjects. Returns rows of (subject, decision, confidence)."""
        rows = []
        for subj in subjects:
            if variant["kind"] == "bool":
                dec, p, rc = SURFACE.decide_bool(variant["domain"], subj, variant["instruction"],
                                                 variant.get("threshold", 0.5))
                rows.append({"subject": subj[:48], "decision": dec, "confidence": round(p, 3)})
            elif variant["kind"] == "choice":
                pick, conf, rc = SURFACE.decide_choice(variant["domain"], subj, variant["instruction"],
                                                       variant["options"])
                rows.append({"subject": subj[:48], "decision": pick, "confidence": round(conf, 3)})
            elif variant["kind"] == "score":
                sc, conf, rc = SURFACE.decide_score(variant["domain"], subj, variant["instruction"],
                                                    variant["options"])
                rows.append({"subject": subj[:48], "decision": round(sc, 2), "confidence": round(conf, 3)})
        return rows

    # smoke: run the binary routing variant over the routing suite
    v1 = next(v for v in VARIANTS if v["name"] == "routing-v1-bool")
    rows = run_variant(v1, SUITES["routing"])
    print("routing-v1-bool over the suite:")
    for r in rows:
        print(f"  strong={r['decision']!s:5} p={r['confidence']:.2f} | {r['subject']}")
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

In [ ]:
if RUN_LIVE:
    queries = SUITES["routing"]
    v1 = next(v for v in VARIANTS if v["name"] == "routing-v1-bool")
    v2 = next(v for v in VARIANTS if v["name"] == "routing-v2-3way")
    v3 = next(v for v in VARIANTS if v["name"] == "routing-v3-score")
    r1 = run_variant(v1, queries)
    r2 = run_variant(v2, queries)
    r3 = run_variant(v3, queries)

    print(f"{'query':46} {'v1-bool':8} {'v2-3way':9} {'v3-score':9}")
    print("-" * 78)
    for q, a, b, c in zip(queries, r1, r2, r3):
        print(f"{q[:44]:46} {str(a['decision']):8} {b['decision']:9} {c['decision']:<9}")

    # where do the structures disagree?
    disagree = sum(1 for a, b in zip(r1, r2)
                   if (a["decision"] is True and b["decision"] == "cheap") or
                      (a["decision"] is False and b["decision"] == "strong"))
    print(f"\nextreme disagreements (bool strong vs 3way cheap, or bool cheap vs 3way strong): {disagree}")
    mid_used = sum(1 for b in r2 if b["decision"] == "mid")
    print(f"3-way used the mid tier on {mid_used}/{len(queries)} queries — "
          f"{'mid tier is doing real work' if mid_used > 0 else 'mid tier is decorative'}")
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

In [ ]:
if RUN_LIVE:
    v1 = next(v for v in VARIANTS if v["name"] == "routing-v1-bool")
    queries = SUITES["routing"]
    print("threshold sweep over the routing suite (escalations = queries sent STRONG):")
    # grab raw p per query once, then sweep thresholds locally (no extra API cost)
    raw_p = []
    for q in queries:
        dec, p, rc = SURFACE.decide_bool("routing", q, v1["instruction"], 0.5)
        raw_p.append((q, p))
    for thr in (0.3, 0.5, 0.7, 0.9):
        esc = sum(1 for _, p in raw_p if p >= thr)
        print(f"  threshold={thr}: {esc}/{len(queries)} escalate to STRONG")
    print("\nraw reported p(needs strong) per query:")
    for q, p in raw_p:
        print(f"  {p:.2f} | {q[:55]}")
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

In [ ]:
if RUN_LIVE:
    vt = next(v for v in VARIANTS if v["name"] == "trust-v1-bool")
    expected = [True, False, True, False]   # 133 is right, 130 wrong, Paris right, Lyon wrong
    rows = run_variant(vt, SUITES["trust"])
    correct = 0
    for subj, r, exp in zip(SUITES["trust"], rows, expected):
        ok = (r["decision"] == exp)
        correct += ok
        print(f"  {'✓' if ok else '✗'} sound={r['decision']!s:5} p={r['confidence']:.2f} | {subj[:52]}")
    print(f"\ntrust-v1-bool accuracy: {correct}/{len(rows)}")
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

In [ ]:
if RUN_LIVE:
    vf = next(v for v in VARIANTS if v["name"] == "fleet-v1-choice")
    rows = run_variant(vf, SUITES["fleet"])
    for r in rows:
        print(f"fleet-v1-choice: node={r['decision']} conf={r['confidence']:.2f}")
        print(f"  task: {r['subject']}")
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')